# Encoder inference: encode thrombus image patches into latent space

## 0. Imports

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
from torch.utils.data import Dataset
from torchvision import transforms

import lightning as pl
import PIL
import numpy as np
import pandas as pd
import argparse
from coding_assets.python import config_manager

## 1. Preparation steps

Declare paths:

In [ ]:
data_root_path = r"C:\Users\flori\OneDrive - univ-angers.fr\Documents\Home\Research\Bureau\data-final-2025june30\validation-data\raw"
reload_checkpoint = r"C:\Users\flori\OneDrive - univ-angers.fr\Documents\Home\Research\Bureau\data-final-2025june30\aQ\last.ckpt"

Prepare pytorch configuration:

In [ ]:
PIL.Image.MAX_IMAGE_PIXELS = 9999999999

# handle communication, especially if working on multiple nodes
try:
    import idr_torch

    using_slurm = True
except:
    using_slurm = False

# Setting the seed
pl.seed_everything(44)

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

LOG_VERSION_FN = 1

## 2. Dataset functions

In [ ]:
# function to load a mask (enables differenciating background from ROIs/regions of interest)
def load_mask(mask_path, masks_downscale_factor):
    mask_pil = PIL.Image.open(mask_path)
    if masks_downscale_factor != 1:
        mask_pil = mask_pil.resize((mask_pil.width // masks_downscale_factor, mask_pil.height // masks_downscale_factor))
    mask_arr = np.array(mask_pil)
    mask_arr = (mask_arr < 255) * 1
    return mask_arr

# convert coordinates from absolute to relative, taking downscale factor into account
def crop_coords_slide2mask(crop_inslide_coords, masks_downscale_factor, mask_size):
    if masks_downscale_factor == 1:
        return crop_inslide_coords
    mask_width, mask_height = mask_size
    inslide_x0, inslide_y0, inslide_x1, inslide_y1 = crop_inslide_coords
    inslide_crop_size = inslide_x1 - inslide_x0
    inmask_crop_size = inslide_crop_size // masks_downscale_factor
    inmask_x0 = inslide_x0 // masks_downscale_factor
    inmask_y0 = inslide_y0 // masks_downscale_factor
    inmask_x1 = inmask_x0 + inmask_crop_size
    inmask_y1 = inmask_y0 + inmask_crop_size
    if inmask_x1 > mask_width:
        offset = inmask_x1 - mask_width
        inmask_x0 = inmask_x0 - offset
        inmask_x1 = inmask_x1 - offset
    if inmask_y1 > mask_height:
        offset = inmask_y1 - mask_height
        inmask_y0 = inmask_y0 - offset
        inmask_y1 = inmask_y1 - offset
    return (inmask_x0, inmask_y0, inmask_x1, inmask_y1)

# calculate proportion of ROI area versus background area in the crop
def check_crop_roi_percentage(mask_arr, crop_inslide_coords, masks_downscale_factor):
    crop_inmask_coords = crop_coords_slide2mask(crop_inslide_coords=crop_inslide_coords, masks_downscale_factor=masks_downscale_factor,
                                                mask_size=mask_arr.shape[::-1])
    x0, y0, x1, y1 = crop_inmask_coords
    roi_pct = mask_arr[y0:y1, x0:x1].sum() / ((x1 - x0) * (y1 - y0))
    return roi_pct

# dataset class to later load crops by batches
class SlideCropsDataset(Dataset):
    def __init__(self, data_element,
                 crop_input_size, slides_downscale_factor, masks_downscale_factor,
                 roi_pct_threshold,
                 transform_func,
                 # seed=44,
                 verbose):
        self.data_element = data_element

        assert type(slides_downscale_factor) is int, "slides_downscale_factor should be of type int"
        assert type(masks_downscale_factor) is int, "masks_downscale_factor should be of type int"
        assert type(crop_input_size) is int, "crop_input_size should be of type int"

        self.slides_downscale_factor = slides_downscale_factor
        self.masks_downscale_factor = masks_downscale_factor
        self.crop_nn_input_size = crop_input_size
        self.crop_inslide_size = slides_downscale_factor * crop_input_size

        self.roi_pct_threshold = roi_pct_threshold

        self.transform_func = transform_func

        # self.seed = seed

        self.im_pil = PIL.Image.open(data_element['image_path'])
        slide_size = self.im_pil.size

        inslide_offset_x = (slide_size[0] % self.crop_inslide_size) // 2
        inslide_offset_y = (slide_size[1] % self.crop_inslide_size) // 2

        inslide_x_starts = inslide_offset_x + np.arange(stop=slide_size[0], step=self.crop_inslide_size)
        inslide_y_starts = inslide_offset_y + np.arange(stop=slide_size[1], step=self.crop_inslide_size)

        mask_arr = load_mask(mask_path=data_element['mask_path'], masks_downscale_factor=masks_downscale_factor)
        self.crop_inslide_coords_list = []
        self.crop_percent_coords_list = []
        for inslide_x_start in inslide_x_starts:
            for inslide_y_start in inslide_y_starts:
                crop_inslide_coords = (inslide_x_start, inslide_y_start, inslide_x_start + self.crop_inslide_size, inslide_y_start + self.crop_inslide_size)
                roi_pct = check_crop_roi_percentage(mask_arr=mask_arr, crop_inslide_coords=crop_inslide_coords,
                                                    masks_downscale_factor=self.masks_downscale_factor)
                if roi_pct >= self.roi_pct_threshold:
                    crop_percent_coords = crop_inslide_coords[0] / slide_size[0], crop_inslide_coords[1] / slide_size[1], crop_inslide_coords[2] / slide_size[
                        0], crop_inslide_coords[3] / slide_size[1]
                    self.crop_inslide_coords_list.append(crop_inslide_coords)
                    self.crop_percent_coords_list.append(crop_percent_coords)

        self.crop_percent_coords_list = np.array(self.crop_percent_coords_list)
        covered_area = len(self.crop_inslide_coords_list) * self.crop_inslide_size ** 2 / (slide_size[0] * slide_size[1])

        if verbose:
            print(f"Kept {len(self.crop_inslide_coords_list)} crops covering {100 * covered_area:.1f}% of the slide area")

    def __len__(self):
        return len(self.crop_inslide_coords_list)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        crop_pil = self.im_pil.crop(self.crop_inslide_coords_list[idx])
        if self.slides_downscale_factor != 1:
            crop_pil = crop_pil.resize((self.crop_nn_input_size, self.crop_nn_input_size))
        label = self.crop_percent_coords_list[idx]

        return self.transform_func(crop_pil), label

# function that will enable loading the patches, without any random augmentation (inference only)
def get_noaugmentation_transform_func():
    transform_func = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    )
    return transform_func

## 3. Deep learning model architecture

In [ ]:
class SimCLR(pl.LightningModule):
    def __init__(self, max_epochs, hidden_dim, lr, temperature, weight_decay, architecture, probe_cv, distributed):
        super().__init__()
        self.save_hyperparameters()
        self.architecture = architecture
        self.sync_dist = distributed
        assert self.hparams.temperature > 0.0, "The temperature must be a positive float!"
        # Base model f(.)
        if architecture == 'resnet18':
            self.convnet = torchvision.models.resnet18(
                num_classes=4 * hidden_dim
            )  # num_classes is the output size of the last linear layer
            # The MLP for g(.) consists of Linear->ReLU->Linear
            self.convnet.fc = nn.Sequential(
                self.convnet.fc,  # Linear(ResNet output, 4*hidden_dim)
                nn.ReLU(inplace=True),
                nn.Linear(4 * hidden_dim, hidden_dim),
            )
        elif architecture == 'resnet50':
            self.convnet = torchvision.models.resnet50(
                num_classes=4 * hidden_dim
            )  # num_classes is the output size of the last linear layer
            # The MLP for g(.) consists of Linear->ReLU->Linear
            self.convnet.fc = nn.Sequential(
                self.convnet.fc,  # Linear(ResNet output, 4*hidden_dim)
                nn.ReLU(inplace=True),
                nn.Linear(4 * hidden_dim, hidden_dim),
            )
        elif architecture == 'vgg16':
            self.convnet = torchvision.models.vgg16(
                num_classes=4 * hidden_dim
            )  # num_classes is the output size of the last linear layer
            # The MLP for g(.) consists of Linear->ReLU->Linear
            self.convnet.classifier = nn.Sequential(
                self.convnet.classifier,  # Linear(ResNet output, 4*hidden_dim)
                nn.ReLU(inplace=True),
                nn.Linear(4 * hidden_dim, hidden_dim),
            )
        else:
            assert False, f"Unsupported architecture: <{architecture}>"

        # probe classifier for validation step
        self.train_losses = []
        self.probe_step_labels = []
        self.probe_step_features = []
        # self.probe_step_projections = []
        self.harmonized_validation_logger = None

    def get_unprojected_features(self, x):  # get features before projection through the mlp g(.) without discarding it
        if self.architecture == 'resnet18' or self.architecture == 'resnet50':
            x = self.convnet.conv1(x)
            x = self.convnet.bn1(x)
            x = self.convnet.relu(x)
            x = self.convnet.maxpool(x)

            x = self.convnet.layer1(x)
            x = self.convnet.layer2(x)
            x = self.convnet.layer3(x)
            x = self.convnet.layer4(x)

            x = self.convnet.avgpool(x)
            x = torch.flatten(x, 1)
            # x = self.convnet.fc(x)
        elif self.architecture == 'vgg16':
            x = self.convnet.features(x)
            x = self.convnet.avgpool(x)
            x = torch.flatten(x, 1)
            # x = self.convnet.classifier(x)

        return x

    def project_features(self, x):
        if self.architecture == 'resnet18' or self.architecture == 'resnet50':
            return self.convnet.fc(x)
        elif self.architecture == 'vgg16':
            return self.convnet.classifier(x)

    def forward(self, batch):
        imgs, labels = batch
        # Forward function that is run when visualizing the graph
        return self.get_unprojected_features(imgs), labels

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.hparams.max_epochs, eta_min=self.hparams.lr / 50
        )
        return [optimizer], [lr_scheduler]

    def training_step(self, batch, batch_idx):
        imgs, _ = batch
        imgs = torch.cat(imgs, dim=0)

        # Encode and project all images
        projections = self.convnet(imgs)
        # Calculate cosine similarity
        cos_sim = F.cosine_similarity(projections[:, None, :], projections[None, :, :], dim=-1)
        # Mask out cosine similarity to itself
        self_mask = torch.eye(cos_sim.shape[0], dtype=torch.bool, device=cos_sim.device)
        cos_sim.masked_fill_(self_mask, -9e15)
        # Find positive example -> batch_size//2 away from the original example
        pos_mask = self_mask.roll(shifts=cos_sim.shape[0] // 2, dims=0)
        # InfoNCE loss
        cos_sim = cos_sim / self.hparams.temperature
        nll = -cos_sim[pos_mask] + torch.logsumexp(cos_sim, dim=-1)
        nll = nll.mean()

        # Logging loss
        self.train_losses.append(nll)
        # self.log(mode + "_loss", nll)
        self.logger.experiment.add_scalar("Loss/Train", nll, self.global_step)

        return nll

## 4. Prepare inference

Prepare hyperparameters for inference:

In [ ]:
# parser.add_argument("--training_augmentation", type=str)
batch_size = 128
crop_input_size = 128
downscale_original_slides_by_factor = 2
downscale_masks_by_factor = 8
roi_pct_threshold = 0.5

trainer_args = {'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
                'devices': 'auto',
                'num_nodes': 1,
                'strategy': 'auto'}

num_workers = 0

Safety checks:

In [ ]:
if crop_input_size is not None:
    assert crop_input_size % downscale_masks_by_factor == 0, "Crop input size must be a multiple of downscale_masks_by_factor"
    assert crop_input_size % downscale_original_slides_by_factor == 0, "Crop input size must be a multiple of downscale_original_slides_by_factor"
    assert crop_input_size % 2 == 0, "Crop size must be a multiple of 2"

print(f'Number of workers is set to {num_workers} while number of available cpus is: {os.cpu_count()}')

## 5. Load and prepare data

In [ ]:
print("Parsing data and training_metadata")

data_elements = []
print(f"Listing elements at root path {data_root_path}")
for slide_name in os.listdir(data_root_path):
    if not os.path.isdir(os.path.join(data_root_path, slide_name)):
        continue
    slide_name_patient = slide_name[:8].upper()
    image_path = os.path.join(data_root_path, slide_name, f"{slide_name}.jpg")
    mask_path = os.path.join(data_root_path, slide_name, "ImResultats", "imFINALE.gif")
    assert os.path.exists(image_path), f"Image file does not exist for {slide_name=}"
    data_elements.append({'patient': slide_name_patient, 'image_path': image_path,
                          'mask_path': mask_path})
print(f"Found {len(data_elements)} elements for {len(np.unique([e['patient'] for e in data_elements]))} unique patients/samples")

In [ ]:
data_elements[:5]

## 6. Load deep learning encoder and run inference

In [ ]:
print("Loading model from checkpoint")

assert os.path.exists(reload_checkpoint) and os.path.isfile(reload_checkpoint), f"Could not locate specified checkpoint: {reload_checkpoint}"

if torch.cuda.is_available():
    print("CUDA available, loading model on GPU")
    model = SimCLR.load_from_checkpoint(reload_checkpoint)
else:
    print("CUDA not available, loading model on CPU")
    model = SimCLR.load_from_checkpoint(reload_checkpoint, map_location=torch.device('cpu'))
model.freeze()

trainer = pl.Trainer(**trainer_args)

Inference loop: would be run for each patient, but we will only look at one patient for this test:

In [ ]:
# for each patient/slide // since this is for debugging purposes, we will only analyze the first patient
i = 1
data_element = data_elements[i]
slide_id = os.path.splitext(os.path.split(data_element['image_path'])[-1])[0]

In [ ]:
print(f"Computing features for slide {i + 1}/{len(data_elements)}: {slide_id}")

# create a dataset for this slide
slide_dataset = SlideCropsDataset(data_element=data_element,
                                  crop_input_size=crop_input_size,
                                  slides_downscale_factor=downscale_original_slides_by_factor,
                                  masks_downscale_factor=downscale_masks_by_factor,
                                  roi_pct_threshold=roi_pct_threshold,
                                  verbose=0,
                                  transform_func=get_noaugmentation_transform_func())

# load slide data
slide_data_loader = data.DataLoader(
    slide_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    pin_memory=True,
    num_workers=num_workers)

print("Running predictions...")

outputs = trainer.predict(model, dataloaders=slide_data_loader)

features = torch.cat([out[0] for out in outputs], axis=0).detach().cpu().numpy().astype(float)
coords = torch.cat([out[1] for out in outputs], axis=0).detach().cpu().numpy().astype(float)

In [ ]:
# show pictures for this patient dataset:
# the code below can be used to show an image of the actual input sent to the model
slide_ds_width = slide_dataset.im_pil.width // downscale_original_slides_by_factor
slide_ds_height = slide_dataset.im_pil.height // downscale_original_slides_by_factor

reconstructed_slide = PIL.Image.new(mode='RGB', size=(slide_ds_width, slide_ds_height), color=0)

inv_normalize = transforms.Normalize(
    mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.255],
    std=[1 / 0.229, 1 / 0.224, 1 / 0.255]
)

from tqdm import tqdm

for crop_data_batch in tqdm(slide_data_loader):
    crops, labels = crop_data_batch
    crops = inv_normalize(crops)
    crops = crops.detach().cpu().numpy()
    crops = crops * 255
    crops = crops.astype(np.uint8)
    crops = [transforms.ToPILImage()(np.moveaxis(crops[j], 0, -1)) for j in range(len(crops))]
    labels = labels.detach().cpu().numpy()
    labels = labels[:, :2]
    labels[:, 0] = labels[:, 0] * slide_dataset.im_pil.width / downscale_original_slides_by_factor
    labels[:, 1] = labels[:, 1] * slide_dataset.im_pil.height / downscale_original_slides_by_factor
    labels = labels.astype(int)
    for crop, crop_coords in zip(crops, labels):
        reconstructed_slide.paste(im=crop, box=list(crop_coords) + [crop_coords[0] + crop_input_size, crop_coords[1] + crop_input_size, ])

    display(reconstructed_slide)

Patch encodings for this patient (for each of the 512 encoded features):

In [ ]:
pd.DataFrame(features, columns=[f"Feature #{f + 1}" for f in range(features.shape[1])], index=[f"Patch #{p + 1}" for p in range(features.shape[0])])

Patch coordinates for this patient (relative to whole slide image):

In [ ]:
pd.DataFrame(coords, columns=["x1", "x1", "x2", "y2"], index=[f"Patch #{p + 1}" for p in range(features.shape[0])])